In [1]:
file1 = "hinglish_conversations.csv"
file2 = "train-00000-of-00001.parquet"

In [2]:
import pandas as pd

df_csv = pd.read_csv(file1)
df_parquet = pd.read_parquet(file2)

In [3]:
df_csv

,input,output
0,"kya yaar,traffic mein stuck ho gaya.","arre, mere ko bhi late hoga ab!"
1,kaunsa movie dekha tune?,"bhai, wo new one, bollywood wala."
2,aaj ka khana kya banaya?,"sabzi aur roti, usual stuff."
3,kal party hai na?,"haan yaar, tu bhi aana."
4,koi plans hai weekend ke?,"nahi yaar, just relaxing at home."
...,...,...
1001318,"Yaar, mujhe YouTube par nature documentaries b...",Nature documentaries? Kitna accha! Lekin kya t...
1001319,Main YouTube par language learning tutorials s...,Language tutorials? Bahut informative! Lekin k...
1001320,"Yaar, mujhe YouTube par mental health discussi...",Mental health? Bahut zaroori hai! Lekin kya lo...
1001321,Main YouTube par eco-friendly living tips shar...,Eco-friendly tips? Bahut accha! Lekin kya tumh...


In [4]:
df_parquet["length"] = df_parquet["messages"].apply(len)

In [5]:
df_parquet.describe()

,length
count,25000.000000
mean,5.641200
std,2.978526
min,3.000000
25%,3.000000
50%,3.000000
75%,9.000000
max,9.000000


In [6]:
df_parquet["messages"][0]

array([{'content': "You are an AI assistant. Provide a detailed answer so user don't need to search outside to understand the answer.\n\nAnswer in mix of hindi and hinglish Translate to hinglish if required.", 'role': 'system'},
       {'content': 'Sports aur national unity ke tarike par ek shayari likh.', 'role': 'user'},
       {'content': 'खेलों की जो बात, वो सब कुछ सिखाते हैं,\nसम्मान, सहयोग, और एकजुटता का संगीत है।\nराष्ट्रीय एकता का प्रतीक हैं, खिलाड़ियों की जीत,\nऐसे मजबूत होते हैं हम, देश के नाम से संघर्ष करते हैं।\n\nऐसे खेल जो दिलों में समानता लाते हैं,\nजहां हर आदमी को एक समान मानते हैं।\nसभी राज्यों के लोग एक साथ खेलकूद कर,\nहमें याद दिलाते हैं, एक भारत, एक संघर्ष।\n\nहंसी, रोशनी, जीत और हार के साथ,\nहमें एकता का अनुभव प्रदान करते हैं।\nस्पोर्ट्स की ये शक्ति, जो देश को जोड़ती है,\nहमें सबको एक निशान देती है, एकता का आदरणीय रंग।\n\nताकि यहाँ बताया जाए, खेलों की इस शक्ति का महत्व,\nराष्ट्रीय एकता को बढ़ावा देते हैं, हर बार।\nहमें एक साथ ले जाते हैं, एक मन, एक आस्था,\nस्पोर्ट्

In [ ]:
df_csv_small = df_csv.sample(n=1_000, random_state=42)

In [8]:
def messagify(row):
    user_msg = row["input"]
    assistant_msg = row["output"]
    message = [{
        "role": "user",
        "content": user_msg,
    }, {
        "role": "assistant",
        "content": assistant_msg,
    }]
    return message

df_csv_small["messages"] = df_csv_small.apply(messagify, axis=1)
df_csv_small

,input,output,messages
485025,Vegetarian recipes mein creativity kaafi zaroo...,"Haaan, main bhindi masala mein coconut daalti ...","[{'role': 'user', 'content': 'Vegetarian recip..."


In [9]:
df_csv_small["length"] = df_csv_small["messages"].apply(len)
df_csv_small.describe()

,length
count,1.0
mean,2.0
std,NaN
min,2.0
25%,2.0
50%,2.0
75%,2.0
max,2.0


In [10]:
df_csv_small.reset_index(drop=True, inplace=True)
df_csv_small["messages"][0]

[{'role': 'user',
  'content': 'Vegetarian recipes mein creativity kaafi zaroori hai, tumhare paas koi unique idea hai?'},
 {'role': 'assistant',
  'content': 'Haaan, main bhindi masala mein coconut daalti hoon, taste aur health dono ke liye perfect!'}]

In [ ]:
df_parquet_small = df_parquet.sample(n=2_000, random_state=42)

In [ ]:
finetune_df = pd.concat(
    [
        df_parquet_small[["messages"]],
        df_csv_small[["messages"]],
    ],
    ignore_index=True
)

finetune_df_train = finetune_df.sample(frac=0.8, random_state=42)
finetune_df_val = finetune_df.drop(finetune_df_train.index)

output_path = "finetune_messages.jsonl"
finetune_df.to_json(output_path, orient="records", lines=True, force_ascii=False)

print(f"Saved {len(finetune_df)} records to {output_path}")

output_path_train = "finetune_messages_train.jsonl"
finetune_df_train.to_json(output_path_train, orient="records", lines=True, force_ascii=False)

output_path_val = "finetune_messages_val.jsonl"
finetune_df_val.to_json(output_path_val, orient="records", lines=True, force_ascii=False)

print(f"Saved {len(finetune_df_train)} training records to {output_path_train}")
print(f"Saved {len(finetune_df_val)} validation records to {output_path_val}")

Saved 3 records to finetune_messages_test.jsonl
Saved 2 training records to finetune_messages_train_test_run.jsonl
Saved 1 validation records to finetune_messages_val_test_run.jsonl
